In [ ]:
from typing import Annotated, List
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core import HumanMessage, SystemMessage

# 01. 출력 스키마 정의
####################################
# 먼저 Section은 보고서의 개별 섹션을 나타내는 구조로, 섹션 제목(name)과 섹션 설명(description)을 포함함.
# Sections 는 이러한 Section 객체들의 목록을 담는 구조로, 전체 보고서의 목차에 해당함. 이와 같이 데이터 구조를 미리 정의해 두면 LLM이 생성해야 할 출력의 형태가 분명해짐.
# 이를 바탕으로, LLM이 자유로운 텍스트가 아니라 정해진 스키마를 따르는 구조화된 결과를 반환하도록 설정할 수 있음.

# 보고서 섹션(Section) 정의
class Section(BaseModel):
    name: str = Field(description="이 보고서 섹션의 제목")
    description: str = Field(descriptin="이 섹션에서 다룰 주요 주제와 개념에 대한 간략한 설명")

# 보고서 섹션 목록 구조. (출력 스키마)
class Sections(BaseModel):
    sections: List[Section] = Field(description="보고서의 각 섹션 목록")

# 아래의 planner 는 LLM에 구조화된 출력(Structured output) 스키마 적용한 객체임.
# 오케스트레이터가 '보고서 목차를 생성해 달라'고 요청하면 결과가 {sections: [...]}형태로 안정적으로 반횐되도록 함.
# 이렇게 하면 문자열을 다시 해석하거나 파싱할 필요없이 생성된 섹션 목록을 곧바로 다음 단계(워커 실행)에 활용할 수 있음.

load_dotenv()

llm = init_chat_model("openai:gpt-4.1")

# LLM 에 구조화 출력(Structured Output) 스키마 적용.
planner = llm.with_structured_output(Sections)

In [ ]:
# 02. 상태 정의.
####################################

# 이제 워크플로우 전반에서 사용할 상태를 정의함.
# State 는 보고서 작성 과정 전체에서 공유되는 전역 상태로, 다음과 같은 정보를 포함함.
#   .topic : 보고서 전체 주제.
#   .sections : 오케스트레이터가 생성한 보고서 섹션 목록
#   .completed_sections: 각 워커가 작성한 섹션 결과를 모아 둔 리스트
#   .final_report : 모든 섹션을 종합해 만든 최종 보고서.

# 특히 completed_sections 는 여러 워커가 생성한 결과를 하나의 리스트로 누적해서 저장해야 하므로 Annotated[list, add] 를 사용해 결과를 자동으로 병합하도록 설정.

# 전체 워크플로 상태.
class State(TypedDict):
    topic: str # 보고서 주제
    sections: list[Section] # 보고서 섹션 목록
    completed_sections: Annotated[list, add] # 모든 워커가 작성한 섹션 결과 모음.
    final_report: str # 최종 보고서

# 워커 상태
class WorkState(TypedDict):
    section: Section
    completed_sections: Annotated[list, add]

In [ ]:
# 03. 노드 정의
####################################

# 이제 보고서 작성 워크플로우를 구성하는 각 노드를 정의함.
# 이 예제에서는 역할에 따라 오케스트레이터, 워커, 종합기(Systhesizer)의 세가지로 나뉨.
# 각 노드는 워크플로우 내에서 명확한 책임을 가지며, 서로 협력해 하나의 보고서를 완성함.

# 오케스트레이터

# 오케스트레이터는 전체 작업을 조율하는 역할을 담당함.
# 입력으로 전달된 보고서 주제를 바탕으로, 보고서를 어떤 구성으로 작성할지에 대한 목차(섹션 기획)를 생성함.
# 즉 실제 내용을 작성하기 전에 '무엇을, 어떤 순서로 쓸 것인지'를 먼저 계획한다는 단계임.

# 아래 함수에서는 구조화된 출력이 적용된 LLM(planner)를 호출해 보고서 주제에 맞는 섹션 목록을 생성함.
# 이때 결과는 자유로운 텍스트가 아니라 앞서 정의한 Sections 스키마를 따르는 형태로 반환되며, 오케스트레이터는 그 중 섹션 목록만을 상태에 저장함.

# 이 단계의 핵심은 보고서를 곧바로 작성하지 않고 작업을 잘게 나눌 기준(섹션)을 먼저 만든다는 점임.
# 이 기준이 이후 워커 노드에 작업을 분배하게 함.

def orchestrator(state: State):
    """보고서 목차를 생성하는 오케스트레이터"""

    report_sections = planner.invoke(
        [
            SystemMessage(content="보고서 목차를 생성해 주세요."),
            HumanMessage(content=f"보고서 주제: {state['topic']}")
        ]
    )

    return {"sections": report_sections.sections}

# 워커

# 워커는 오케스트레이터가 생성한 섹션 중 하나를 맡아 실제 내용을 작성하는 역할을 함.
# 각 워커는 전체 보고서 아닌, 자신에게 할당된 섹션 정보만을 입력으로 받음.

# 아래 함수에서는 섹션의 제목과 설명을 프롬프트에 포함돼 LLM을 호출하고, 해당 섹션의 본문을 생성함.
# 이때 '머리말을 포함하지 말것', '마크다운 형식을 사용할 것'과 같은 지침을 함께 전달함으로써 여러 섹션을 나중에 하나로 합쳤을 때도 형식이 일관성을 띠게 함.

# 워커는 작성 결과를 completed_sections 라는 리스트에 담아 반환함.

def llm_call(state: WorkState):
    """각 보고서 섹션을 작성하는 워커"""

    section = llm.invoke([
        SystemMessage(content="제공된 제목과 설명에 따라 보고서 섹션을 작성하세요. 각 섹션에 머리말은 포함하지 말고, 마크다운 형식을 사용하세요."),
        HumanMessage(content=f"섹션 제목: {state['section'].name}\n 설명: {state['section'].description}")
    ])

    return {"completed_sections": [section.content]}

# 종합기(Synthesizer)

# 종합기는 모든 워크가 작성한 섹션 결과를 하나로 합쳐 최종 보고서를 만드는 역할을 함.
# 이 노드는 새로운 내용을 생성하기 보다는 이미 생성된 섹션들을 정리하고 결합하는데 집중함.

# 아래 예제에서는 각 섹션 사이에 마크다운 구분선(---)을 삽입해  최종 결과가 읽기 좋은 형태가 되도록 구성함.
def synthesizer(state: State):
    """작성된 섹션들을 종합하여 최종 보고서 생성"""

    completed_sections = state["completed_sections"]
    combined_report = "\n===\n".join(completed_sections)
    return {"final_report": combined_report}

In [ ]:
# 04. 워커 할당.(Conditional Edge)

# 이제 오케스트레이터가 생성한 섹션 목록을 바탕으로, 섹션 마다 하나씩 워커를 동적으로 생성해야 함. 이를 위해 assign_workers 함수를 정의함.
# 이 함수는 상태에 저장된 섹션 목록을 순회하며, 각 섹션을 입력으로 사용하는 llm_call 노드를 실행하라는 지시(Send)를 생성함. 섹션이 많을 수록 워커 작업도 그 만큼 많이 생성됨.

def assign_workers(state: State):
    """각 섹션에 워커 할당"""
    return [Send("llm_call", {"section": s}) for s in state["sections"])]

In [ ]:
# 05. 워크플로우 구성.
# 이제 앞에서 정의한 노드들을 하나의 실행 흐름으로 연결함.
# StateGraph 를 통해 노드간의 실행 순서와 관계를 그래프 형태로 정의함.
# 이 그래프는 '어떤 노드가 언제 실행되는가' 뿐만 아니라, 실행 결과에 따라 다음 단계가 어떻게 달라지는지까지 함께 표현함.

# 먼저 StateGraph(State)를 사용해 워크플로우 빌더를 생성함.
# 이는 이 그래프가 어떤 상태 구조를 기준으로 동작하는지를 선언하는 단계임. 다음으로, 오케스트레이터, 워커, 종합기 노드를 그래프에 등록함.
# 그리고 노드 간의 실행 흐름을 엣지로 연결함.

# 워크플로우 구성은 먼저 그래프의 시작 지점(START)에서 오케스트레이터 노드로 이동하도록 설정함.
# 오케스트레이터가 실행된 이후에는 생성된 섹션 목록을 바탕으로 여러개의 워커를 실행함.
# 이를 위해 add_conditional_edges 를 사용해 오케스트레이터의 결과에 따라 assign_workers함수가 반환하는 워크 실행 지시를 그래프에 반영함.
# 이 단계에서 섹션의 개수만큼 llm_call 노드가 생성되어 실행함. 각 워커가 섹션 작성을 마치면 그 결과는 종합기 노드로 전달함.
# 종합기는 모든 워커의 결과가 모인 이후에 실행되어 섹션들을 하나의 최종 보고서를 결합함.

# 모든 엣지와 노드가 정의 되면 compile() 를 호출해 그래프를 실행 가능한 워크플로우를 변환함.

# 워크플로우 빌더 생성.
builder = StateGraph(State)

# 노드 등록.
builder.add_node("orchestrator", orchestrator)
builder.add_node("llm_call", llm_call)
builder.add_node("synthesizer", synthesizer)

# 엣지 연결.
builder.add_edge(START, "orchestrator")
builder.add_conditional_edges("orchestrator", assign_workers, ["llm_call"])
builder.add_edge("llm_call", "synthesizer")
builder.add_edge("synthesizer", END)

# 컴파일
orchestrator_worker = builder.compile()

In [ ]:
# 06. 워크플로우의 실행
# 먼저 draw_mermaid_png() 를 사용해 그래프 구조를 다이어그램으로 시각화함. 이를 통해 오케스트레이터-워커-종합기 로 이어지는 흐름을 한눈에 파악할 수 있음.

# 다이어그램 출력.
display(Image(orchestrator_worker.get_graph().draw_mermaid_png()))

# 실행
state = orchestrator_worker.invoke({"topic": "랭그래프 오케스트레이션-워커 법칙에 관한 보고서"})
print(state["final_report"])

# 오케스트레이터-워커 방식의 장점은 복잡하고 구조가 유동적인 작업을 입력에 맞춰 동적으로 분할할 수 있다는 점임.
# 병렬 처리는 사전에 작업을 정의해 동시에 실행하는 한편, 오케스트레이터-워커 는 실행 시점에 필요한 작업을 결정함.
# 다만 전체 결과 품질은 오케스트레이터 단계의 품질에 크게 좌우되므로 이 단계의 프롬프트 설계가 특히 중요함.